## Paper Analysis

### Decoding: Multivariate Pattern Analysis of EEG Signals

The core analysis in Moerel et al. (2025) uses time-resolved multivariate pattern analysis (MVPA) to determine whether the spatial pattern of EEG activity across 64 channels contains information about the players' decisions at each moment during a trial. Rather than analysing individual channels in isolation, MVPA treats the full set of channel voltages at each time point as a high-dimensional pattern and asks: can a classifier learn to distinguish between Rock, Paper, and Scissors responses based on these patterns? This approach has become standard in cognitive neuroscience because it is sensitive to distributed neural representations that univariate methods would miss (Grootswagers et al., 2017).


#### Phase-wise Epoching & Baseline Correction 



Each trial in the experiment consists of three temporally distinct phases: Decision (0–2 s), Response (2–4 s), and Feedback (4–5 s). Critically, the authors do not treat the full 5-second trial as a single epoch for the decoding. Instead, they split each trial into three separate sub-epochs, each locked to the onset of its respective phase:

- **Part A (Decision):** –0.2 to 2.0 s relative to trial onset
- **Part B (Response):** 1.8 to 4.0 s relative to trial onset, then time-shifted so that 0 corresponds to the Response phase onset at 2.0 s
- **Part C (Feedback):** 3.8 to 5.0 s relative to trial onset, then time-shifted so that 0 corresponds to the Feedback phase onset at 4.0 s

The reason for this splitting is that each phase involves a qualitatively different cognitive process: during Decision the participant forms their choice, during Response they execute a button press, and during Feedback they receive visual information about both players' choices. If the entire 5-second epoch were baseline-corrected as a single unit, activity from earlier phases would bleed into the baseline estimate for later phases. By treating each phase independently, the baseline correction can be applied relative to the onset of each phase, ensuring that the decoding results for each phase reflect only the neural processes occurring within that phase.

The 200 ms of overlap (e.g., Part A ends at 2.0 s while Part B starts at 1.8 s) exists specifically to provide each sub-epoch with a pre-phase baseline window of –0.2 to 0 s in its own shifted time frame. 

For each of the three sub-epochs, the authors apply baseline correction using the 200 ms window preceding the phase onset (–0.2 to 0 s in the shifted time frame). This involves subtracting the mean voltage across the baseline window from every time point in the sub-epoch, separately for each channel and trial.

Baseline correction serves two purposes here. First, it removes slow voltage drifts that could differ between trials, which would otherwise add noise to the decoding and reduce sensitivity. Second, and more importantly in this context, it ensures that the classifier is decoding phase-specific neural activity rather than carry-over activity from the preceding phase. Without per-phase baseline correction, above-chance decoding during the Response phase could partly reflect residual Decision-phase activity rather than genuine Response-phase encoding.

#### Averaging into 250 ms time bins


After baseline correction, the continuous EEG data within each sub-epoch is averaged into non-overlapping 250 ms time bins. This produces 8 bins for the Decision phase (0–2 s), 8 bins for the Response phase (0–2 s in shifted time), and 4 bins for the Feedback phase (0–1 s in shifted time), totalling 20 time bins per trial.

Time-binning serves a dual purpose. First, it reduces the dimensionality of the temporal axis, making the subsequent classification more computationally tractable. Second, averaging within 250 ms windows increases the signal-to-noise ratio of each data point by smoothing out high-frequency noise that is unlikely to carry decision-related information. The choice of 250 ms is a standard compromise in the EEG decoding literature: narrow enough to preserve the temporal dynamics of decision-making (which unfolds over hundreds of milliseconds), but wide enough to average out noise effectively. The authors deliberately chose not to apply temporal filtering to the data, as filtering can introduce temporal smearing artifacts that distort the time course of decoded information (van Driel et al., 2021; Delorme, 2023). Time-binning achieves a similar noise-reduction effect without introducing such artifacts.

#### Removal of block-boundary trials


The experiment consisted of 12 blocks of 40 trials each. The first trial of each block was excluded from the decoding analysis because there is no preceding trial within the same block to provide a valid "previous response" label for decode targets 3 and 4. This removal is applied to all four decode targets for consistency, resulting in 468 trials per participant (480 minus 12 block-initial trials).


#### Pseudo-trial construction


Before classification, the authors construct pseudo-trials by averaging together small groups of 4 real trials that share the same response class and cross-validation fold. This averaging is repeated 20 times with different random groupings (using CoSMoMVPA's `cosmo_average_samples` function with `'count', 4, 'repeats', 20, 'seed', 1`), producing 20 pseudo-trials per class per fold.

This step is motivated by two considerations. First, averaging increases the signal-to-noise ratio: single EEG trials are extremely noisy, and averaging 4 trials together roughly doubles the SNR (since noise scales with $\sqrt{n}$ while signal scales linearly). This makes the subtle decision-related patterns more detectable by the classifier. Second, pseudo-trial construction naturally balances the number of samples per class. In the raw data, the three response classes (Rock, Paper, Scissors) may have unequal trial counts due to participant biases or no-response trials. After pseudo-trial construction, each class has exactly 20 pseudo-trials per fold, eliminating class imbalance that could bias the classifier.

The balanced sampling scheme (implemented via `cosmo_sample_unique`) ensures that across all 20 repeats, each original trial is used approximately the same number of times, so no single trial disproportionately influences the result.

#### Classifier: Regularised Linear Discriminant Analysis




The authors use Linear Discriminant Analysis (LDA) with a fixed regularisation parameter of $\lambda = 0.01$ as their classifier (CoSMoMVPA's `cosmo_classify_lda`). LDA models each class as a multivariate Gaussian distribution with a shared covariance matrix and classifies test samples based on which class centroid they are closest to in Mahalanobis distance. The regularisation adds a small multiple of the identity matrix to the estimated covariance, specifically:

$$\Sigma_{\text{reg}} = \Sigma + \lambda \cdot \frac{\text{trace}(\Sigma)}{p} \cdot I$$

where $p$ is the number of features (channels). This is an additive* regularisation: the identity matrix, scaled by the average eigenvalue, is added to the covariance.

LDA is a well-motivated choice for this task for several reasons (Grootswagers et al., 2017; Guggenmos et al., 2018):

1. **Efficiency with small samples:** With 600 total pseudo-trials (3 classes × 10 folds × 20 repeats), training in each fold uses about 540 samples against 64 features (sample-to-feature ratio of roughly 8:1). LDA's closed-form solution makes it well-suited for this regime, unlike neural networks which would require considerably more data to avoid overfitting.
2. **Regularisation necessity:** With 64 channels and ~540 training samples per fold, the raw 64×64 covariance matrix estimate is poorly conditioned. The regularisation stabilises the matrix inversion by shrinking it towards a scaled identity, preventing the classifier from fitting noise in the covariance structure.
3. **Linear assumption:** At the temporal resolution of 250 ms EEG bins, the neural patterns distinguishing Rock, Paper, and Scissors are expected to differ primarily in their mean spatial distributions rather than in complex nonlinear relationships. A linear classifier is therefore appropriate and avoids unnecessary model complexity.
4. **Low computational cost:** LDA has a closed-form solution that requires only computing class means and a shared covariance matrix. This is important because the analysis involves running 20 time bins × 10 folds × 4 decode targets × 62 participants = 49,600 classifier fits for the temporal decoding alone, and considerably more for the searchlight.

#### Cross-validation



Classification is performed using 10-fold cross-validation. The pseudo-trials are divided into 10 folds (chunks) such that each fold contains roughly equal numbers of each response class. In each iteration, the classifier is trained on 9 folds and tested on the remaining fold. Accuracy is computed as the total number of correct predictions across all folds divided by the total number of test samples. With 3 classes, chance-level accuracy is 33.33%.

The chunk assignment is performed by CoSMoMVPA's `cosmo_chunkize` function, which distributes trials across folds in a balanced manner. Importantly, the pseudo-trial averaging is performed within each fold (not across folds), so training and test pseudo-trials are constructed from non-overlapping sets of original trials. This prevents information leakage between training and test sets.

#### Decoding targets






The analysis decodes four targets, each providing different information about the decision-making process:

1. **Own response (current trial):** Whether the player chose Rock, Paper, or Scissors. Above-chance decoding indicates that the EEG pattern carries information about the participant's own decision.
2. **Opponent's response (current trial):** Whether the opponent chose Rock, Paper, or Scissors. Above-chance decoding during the Decision and Response phases would suggest that the participant can predict their opponent's move; during Feedback it reflects the visually presented outcome information.
3. **Own previous response:** The player's choice on the preceding trial. Above-chance decoding suggests that the brain maintains a representation of the previous action, which could reflect a strategy (e.g., win-stay, lose-shift).
4. **Opponent's previous response:** The opponent's choice on the preceding trial. Above-chance decoding indicates that the participant encodes the opponent's past behaviour, potentially to inform their current decision.

#### Channel searchlight






In addition to the temporal decoding (which uses all 64 channels), the authors perform a channel searchlight analysis to identify which brain regions contribute to the decoding. For each channel, a small neighbourhood is constructed consisting of the channel itself and its 4 nearest neighbours (determined by Euclidean distance between electrode positions), giving 5 features per searchlight location. The same cross-validated LDA decoding is then performed using only this subset of channels. The result is a topographic map of decoding accuracy for each time bin, which can be visualised as a scalp map.

This approach reveals whether the decoded information is driven by localised brain activity (e.g., posterior channels for visual feedback processing) or distributed patterns across the scalp (e.g., for decision-related activity). The paper reports that Decision and Response phase decoding shows distributed topographies, consistent with decision-related processes, while Feedback phase decoding shows a posterior focus, consistent with visual processing of the displayed outcome.

## Paper Reproduction

### Decoding Pipeline



The original decoding analysis was implemented in MATLAB using the FieldTrip (version 20240110) and CoSMoMVPA (version 1.1.0) toolboxes. Our reproduction reimplements this pipeline in Python using MNE-Python and NumPy, with custom implementations of several CoSMoMVPA functions where no direct equivalent exists in the Python ecosystem. We describe each step of the reproduction, highlighting where custom code was necessary and what discrepancies we identified along the way.


#### Phase splitting and baseline correction



MNE-Python provides built-in methods for cropping epochs (`epochs.crop()`) and applying baseline correction (`epochs.apply_baseline()`). However, using these directly would not replicate the MATLAB pipeline faithfully. The MATLAB code manually selects time windows from the continuous epoch data using logical indexing on the time vector, shifts the time labels by subtracting the phase onset, and then applies baseline correction on the shifted time axis. MNE's `crop` method, by contrast, operates on the original time axis and does not support the time-shifting step that is needed before baseline correction of Parts B and C.

We therefore implemented the phase splitting and baseline correction manually using NumPy array operations. For each of the three parts (Decision, Response, Feedback), we select the relevant time window from the full epoch array, shift the time labels so that 0 corresponds to the phase onset, and subtract the mean of the –0.2 to 0 s baseline window. This approach matches the MATLAB code line-for-line: the same time boundaries (–0.2 to 2.0 s for Part A, 1.8 to 4.0 s for Part B, 3.8 to 5.0 s for Part C), the same time shifts (subtract 2.0 s for Part B, 4.0 s for Part C), and the same baseline window (–0.2 to 0 s in shifted coordinates).


#### Custom LDA classifier



This was the most critical implementation decision in the entire reproduction. Scikit-learn provides `LinearDiscriminantAnalysis` with a `shrinkage` parameter, which would appear to be a straightforward replacement for CoSMoMVPA's `cosmo_classify_lda`. However, after consulting the CoSMoMVPA source code, we discovered that the two implementations use different regularisation formulas.

CoSMoMVPA computes the regularised covariance as:

$$\Sigma_{\text{reg}} = \Sigma + \lambda \cdot \frac{\text{trace}(\Sigma)}{p} \cdot I$$

where $\Sigma$ is the pooled within-class covariance normalised by $N$ (the total number of training samples), $\lambda = 0.01$ is the regularisation parameter, $p$ is the number of features, and $I$ is the identity matrix. This is an additive regularisation: the identity matrix scaled by the average eigenvalue is added to the covariance, while $\Sigma$ itself is left unchanged.

Scikit-learn's shrinkage LDA, by contrast, uses a convex combination:

$$\Sigma_{\text{reg}} = (1 - \alpha) \cdot \Sigma + \alpha \cdot \frac{\text{trace}(\Sigma)}{p} \cdot I$$

where $\alpha$ is the shrinkage intensity. Both toolboxes normalise the pooled within-class covariance by $N$ (the total number of training samples), so the covariance estimates themselves are identical. The difference lies solely in how the regularisation is applied: CoSMoMVPA adds a scaled identity to the full covariance, while scikit-learn interpolates between the covariance and a scaled identity. With $\lambda = \alpha = 0.01$, the additive formula gives $\Sigma + 0.01 \cdot \mu \cdot I$ while the convex formula gives $0.99 \cdot \Sigma + 0.01 \cdot \mu \cdot I$ (where $\mu = \text{trace}(\Sigma)/p$). The practical effect is that scikit-learn slightly down-weights the off-diagonal covariance structure as $\alpha$ increases, whereas CoSMoMVPA preserves it entirely and only inflates the diagonal. At $\lambda = 0.01$ this difference is small, but it is nonzero and accumulates across the 49,600+ classifier fits in the full analysis.

We therefore wrote a custom LDA implementation that matches the CoSMoMVPA source code exactly:

1. Compute per-class means
2. Compute pooled within-class scatter matrix: $S_w = \sum_k (X_k - \mu_k)^T (X_k - \mu_k)$
3. Normalise by $N$ (total training samples): $\Sigma = S_w / N$
4. Add regularisation: $\Sigma_{\text{reg}} = \Sigma + 0.01 \cdot \frac{\text{trace}(\Sigma)}{p} \cdot I$
5. Compute class weights: $W = \mu \cdot \Sigma_{\text{reg}}^{-1}$
6. Compute bias: $b_k = \sum_j W_{kj} \cdot \mu_{kj}$
7. Predict: $\hat{y} = \arg\max_k \left( x \cdot W_k^T - 0.5 \cdot b_k \right)$

This is worth discussing because it illustrates a broader point about reproduction studies: even when two implementations claim to perform "regularised LDA," the specific regularisation formula can differ between toolboxes, and these differences can affect results. In our case, using scikit-learn's shrinkage would have produced similar but not identical results to the original paper.

#### Custom `cosmo_sample_unique` for balanced pseudo-trial averaging



CoSMoMVPA's `cosmo_average_samples` function internally calls `cosmo_sample_unique` to generate balanced sampling indices. This function ensures that across all 20 repeats, each original trial is used approximately the same number of times, unlike simple random sampling, where some trials might be used many times while others are never selected.

The algorithm works by:

1. Generating (`repeats` + 1) random permutations of the trial indices and concatenating them column-wise into a flat vector
2. Walking through this vector with a position pointer to fill each repeat column with `count` values, skipping any position that has already been consumed globally or whose value already appears in the current column (preventing within-column duplicates)
3. Sorting each column of the resulting index matrix

The extra permutation (repeats + 1 rather than repeats) provides overflow buffer so the walk does not run out of candidates. We ported this algorithm directly from the CoSMoMVPA MATLAB source code to Python, including the column-major flattening (`ravel(order='F')`) to match MATLAB's memory layout. A simpler approach, such as calling `rng.choice(indices, size=count, replace=False)` independently for each repeat, would not guarantee balanced usage across repeats and would produce different pseudo-trials.

#### Channel searchlight

For the channel searchlight, the original MATLAB code uses `cosmo_meeg_chan_neighborhood(ds, 'count', 4)` to define a neighbourhood of exactly 4 nearest channels for each searchlight centre. We replicate this by computing a pairwise Euclidean distance matrix from the electrode positions (obtained from the MNE montage), then for each channel selecting the 4 channels with the smallest distances. Combined with the centre channel itself, this gives 5 features per searchlight location, matching the original.

One subtlety here is the coordinate space. Since we select neighbours by rank order (the 4 closest) rather than by a distance threshold, the absolute scale of the coordinates does not affect the neighbour selection, the ranking is preserved under any uniform scaling. In practice, this leads to similar results as the original MATLAB code, where the neighbourhood distance threshold was 0.99 while all the coordinates were within a unit circle. In other words, there would always be more than 4 neighbours close enough to be considered. 

## Our Contributions


### Linear Decoding: Comparing Classifiers Beyond LDA

#### Motivation

The original paper uses a single classifier (regularised LDA with $\lambda = 0.01$) for all decoding analyses. While LDA is a standard and well-motivated choice, a natural question is whether the results depend on this particular classifier. If the decoded neural information is genuinely present in the EEG signal, it should be recoverable by different linear classifiers, each of which makes different assumptions about the data and uses different optimisation objectives. Conversely, if the results were an artefact of LDA's specific regularisation or its sensitivity to particular covariance structures, alternative classifiers might yield qualitatively different temporal profiles.


#### Implementation

To test this, we ran the identical decoding pipeline (same pseudo-trials, same cross-validation folds, same time bins) with four linear classifiers:

1. **LDA (CoSMoMVPA-style):** Our custom reimplementation matching the original paper's regularised LDA with $\lambda = 0.01$ additive regularisation. This serves as the reproduction baseline.

2. **Shrinkage LDA (Ledoit-Wolf):** Scikit-learn's `LinearDiscriminantAnalysis` with `shrinkage='auto'`, which analytically computes the optimal shrinkage intensity using the Ledoit-Wolf estimator rather than relying on a fixed $\lambda$. The key difference from the CoSMoMVPA variant is that the regularisation strength adapts to each training set: when the sample covariance is well-conditioned, shrinkage is minimal; when it is poorly conditioned, shrinkage increases automatically. This makes the classifier less sensitive to the arbitrary choice of $\lambda = 0.01$. Additionally, as discussed in the reproduction section, scikit-learn uses a convex combination formula ($\Sigma_\text{reg} = (1-\alpha)\Sigma + \alpha \cdot \mu I$) rather than the additive one, meaning it slightly down-weights the off-diagonal covariance structure as shrinkage increases.

In [ ]:
def classify_shrinkage_lda(train_data, train_labels, test_data):
    clf = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    clf.fit(train_data, train_labels)
    return clf.predict(test_data)

3. **Linear SVM:** A linear support vector machine (`LinearSVC` with $C = 1.0$) that finds the maximum-margin separating hyperplane between classes. Unlike LDA, which models the class-conditional distributions and finds the Bayes-optimal boundary under Gaussian assumptions, SVM directly optimises the decision boundary without modelling the full distributions. This makes SVM potentially more robust when the Gaussian assumption is violated (e.g., when EEG voltage distributions have heavier tails than a Gaussian). Because SVM's hinge loss is sensitive to feature scale, we z-score the features using the training set mean and standard deviation before fitting, applying the same transformation to the test set.

In [ ]:
def classify_svm(train_data: np.ndarray, train_labels: np.ndarray, test_data: np.ndarray) -> np.ndarray:
    """Linear SVM with z-scored features."""
    train_scaled, test_scaled = zscore(train_data, test_data)
    clf = LinearSVC(C=1.0, max_iter=10000, dual="auto")
    clf.fit(train_scaled, train_labels)
    return clf.predict(test_scaled)

4. **Logistic Regression (L2):** Multinomial logistic regression (`LogisticRegression` with $C = 1.0$, L-BFGS solver) that minimises the cross-entropy loss with L2 regularisation. Like SVM, it directly optimises a discriminative objective rather than modelling class distributions. Logistic regression offers an interesting middle ground: like LDA, it produces probabilistic class predictions, but like SVM, it does not assume Gaussian class-conditional distributions. Feature standardisation is essential here: without z-scoring, channels with large voltage ranges dominate the gradient, effectively causing the optimiser to ignore smaller channels and predict the majority class uniformly, producing chance-level accuracy.

In [ ]:
def classify_logreg(train_data: np.ndarray, train_labels: np.ndarray, test_data: np.ndarray) -> np.ndarray:
    """L2-regularised multinomial logistic regression with z-scored features."""
    train_scaled, test_scaled = zscore(train_data, test_data)
    clf = LogisticRegression(C=1.0, max_iter=10000, solver="lbfgs")
    clf.fit(train_scaled, train_labels)
    return clf.predict(test_scaled)

A note on feature standardisation: LDA (both variants) is inherently invariant to affine feature transformations, because the covariance matrix captures the relative scaling of features. SVM and logistic regression are not, so their loss functions treat all features equally in the raw input space. Channels with larger voltage ranges would dominate the decision boundary unless the features are standardised first. The z-scoring is computed from the training set and applied to both training and test data within each cross-validation fold, preventing any information leakage.

#### Results

The four figures show the temporal decoding accuracy for each classifier across the four decode targets (own response, opponent's response, own previous response, opponent's previous response). The dashed line at 33.3% marks theoretical chance for 3-class classification, and the shaded bands show the 95% confidence interval across participants.

**Own response (panel a).** All four classifiers recover the same temporal profile: accuracy rises modestly above chance during the Decision phase (~34–35%), peaks sharply in the early Response phase (~38%), drops back to just above change by late Response, rand then peaks less sharply during Feedback (~36%). The peak during the Response phase corresponds to the moment participants execute their button press, when the neural representation of the chosen action is strongest. The fact that accuracy is already above chance during the Decision phase confirms the original paper's finding that participants begin forming their decision during this preparatory period. All four classifiers agree on this profile, with confidence intervals overlapping at every time bin.

**Opponent's response (panel b).** All classifiers show accuracy at or below chance during the Decision and Response phases, rising sharply only during Feedback when the opponent's choice is visually displayed. The Feedback-phase peak (~38–39%) is visually driven, and all four classifiers detect it equally well. The absence of above-chance decoding during Decision and Response confirms that participants cannot reliably predict their opponent's next move, consistent with the behavioural unpredictability discussed in the paper.

**Own previous response (panel c).** Decoding accuracy for the player's own choice on the preceding trial is modestly above chance during the Decision phase (~34–35%) and the Response phase (~35%), hovering close to the chance line during Feedback. This subtle effect suggests that a trace of the previous action persists into the current trial's decision period. All four classifiers capture this pattern with similar magnitude and timing, though the effect is small relative to the confidence intervals.

**Opponent's previous response (panel d).** All classifiers show an early Decision-phase peak (~36–37% at the first time bin), after which accuracy gradually returns toward chance. This early peak suggests that participants encode what their opponent played on the previous trial at the very start of the new decision period, potentially reflecting a strategy of incorporating the opponent's recent behaviour. Again, the temporal profile is consistent across all four classifiers.

<img src="images/LDA.png" width="400"> <img src="images/SVM.png" width="400">

<img src="images/LOGREG.png" width="400"> <img src="images/ShrinkageLDA.png" width="400">

#### Interpretation

The most striking result of this comparison is the convergence: all four classifiers produce virtually indistinguishable temporal profiles for every decode target. This convergence is informative for several reasons.

First, it demonstrates that the original paper's findings are robust to classifier choice. The decoded neural information is not an artefact of LDA's specific regularisation formula or its Gaussian distributional assumptions. Whether we use a generative model (LDA), a discriminative margin-based model (SVM), or a discriminative probabilistic model (logistic regression), the same temporal structure emerges. This strengthens confidence that the decoding reflects genuine neural representations rather than classifier-specific biases.

Second, the convergence tells us something about the nature of the neural signal. When different linear classifiers with different loss functions (quadratic, hinge, cross-entropy) all achieve the same accuracy, it suggests that the discriminative information resides primarily in the class means (the spatial distribution of average voltages across channels for Rock vs Paper vs Scissors) rather than in the covariance structure. LDA explicitly models the covariance, while SVM and logistic regression ignore it. So, the fact that they perform equally well implies that the covariance plays a minor role in separating the classes. In other words, the EEG patterns distinguishing the three responses differ mainly in where activity is stronger or weaker across the scalp, not in how activity co-varies between channels.

Third, the similar performance of the fixed-regularisation LDA ($\lambda = 0.01$) and the data-adaptive Shrinkage LDA (Ledoit-Wolf) suggests that the fixed regularisation chosen by the original authors was already near-optimal for this dataset. The Ledoit-Wolf estimator, which analytically minimises the expected squared error between the shrunk and true covariance, provides no measurable advantage. This reassures that the original results are not sensitive to the specific regularisation parameter, and a different (reasonable) choice of $\lambda$ would likely have produced similar findings.

Finally, the fact that z-scored SVM and logistic regression match the LDA results despite the additional preprocessing step (feature standardisation) suggests that there are no channels with extreme voltage ranges that disproportionately drive the LDA results. If a few high-variance channels had been dominating the LDA covariance estimate, the z-scored classifiers (which equalise channel contributions) would have shown different accuracy profiles. The consistency indicates that the neural information is distributed across channels of comparable variance, which aligns with the searchlight topographies reported in the paper showing spatially distributed encoding.